In [1]:
from pynq import Overlay
import time
import numpy as np
import sys

# Load the Overlay (Configures the Bitstream)
print("Loading Overlay...")
overlay = Overlay("riscv_pynq_wrapper.bit")

# run on the board and set up
instr_mem = overlay.bram_controller_irom
data_mem  = overlay.bram_controller_dram
reset_gpio = overlay.axi_gpio_reset  

print("Overlay Loaded Successfully!")

Loading Overlay...


Overlay Loaded Successfully!


In [2]:
def try_write_read(mem, offset, val):
    try:
        mem.write(offset, val)
        r = mem.read(offset)
        print(f"{mem}: offset 0x{offset:x} wrote 0x{val:08x} read 0x{r:08x}")
        return r
    except Exception as e:
        print("ERR", e)

# try offsets (bram uses word aligned access)
for off in [0, 1, 4, 16, 0x100]:
    try_write_read(instr_mem, off, 0xdeadbeef)

# simple pattern to check endianness (usually little endian)
try_write_read(instr_mem, 0, 0x01020304)
try_write_read(instr_mem, 0, 0x11111111)
try_write_read(instr_mem, 0, 0xffffffff)
try_write_read(instr_mem, 0, 0x00000000)

# same tests on DRAM controller
for off in [0, 4]:
    try_write_read(data_mem, off, 0xdeadbeef)
    try_write_read(data_mem, off, 0x01020304)


<pynq.pl_server.embedded_device.EmbeddedXrtMemory object at 0xb48f6370>: offset 0x0 wrote 0xdeadbeef read 0xdeadbeef
ERR Unaligned write: offset must be multiple of 4.
<pynq.pl_server.embedded_device.EmbeddedXrtMemory object at 0xb48f6370>: offset 0x4 wrote 0xdeadbeef read 0xdeadbeef
<pynq.pl_server.embedded_device.EmbeddedXrtMemory object at 0xb48f6370>: offset 0x10 wrote 0xdeadbeef read 0xdeadbeef
<pynq.pl_server.embedded_device.EmbeddedXrtMemory object at 0xb48f6370>: offset 0x100 wrote 0xdeadbeef read 0xdeadbeef
<pynq.pl_server.embedded_device.EmbeddedXrtMemory object at 0xb48f6370>: offset 0x0 wrote 0x01020304 read 0x01020304
<pynq.pl_server.embedded_device.EmbeddedXrtMemory object at 0xb48f6370>: offset 0x0 wrote 0x11111111 read 0x11111111
<pynq.pl_server.embedded_device.EmbeddedXrtMemory object at 0xb48f6370>: offset 0x0 wrote 0xffffffff read 0xffffffff
<pynq.pl_server.embedded_device.EmbeddedXrtMemory object at 0xb48f6370>: offset 0x0 wrote 0x00000000 read 0x00000000
<pynq.pl_s

In [3]:
print("Putting Core into RESET")
# Write 0 to the GPIO (resetn) to HALT the core
reset_gpio.channel1.write(0x1, 0x0)

# ... inside the Python script ...

print("MEMORY SANITY CHECK")

# Use an address offset far away from 0
test_addr = 0x100  # Offset 256 bytes
test_pattern = 0xCAFEBABE

print(f"Writing 0x{test_pattern:08x} to Instruction Memory at Address 0x{test_addr}...")
instr_mem.write(test_addr, test_pattern)

read_back = instr_mem.read(test_addr)
print(f"Read back value: 0x{read_back:08x}")

# result
if read_back == test_pattern:
    print("Memory Read/Write is perfect.")
elif read_back == 0x000000EF:
    print("FAILURE: Byte-Write Issue Detected.")
    # We stop here because loading the program would be pointless
    sys.exit() 
elif read_back == 0x00000000:
    print("FAILURE: Zero Readback.")
    print("Diagnosis: The write didn't 'stick'.")
    sys.exit() 
else:
    print("FAILURE: Unknown Data Corruption.")
    print(f"Expected: 0x{test_pattern:08x}, Got: 0x{read_back:08x}")
    sys.exit()

print("SANITY CHECK PASSED!\n")

Putting Core into RESET
MEMORY SANITY CHECK
Writing 0xcafebabe to Instruction Memory at Address 0x256...
Read back value: 0xcafebabe
Memory Read/Write is perfect.
SANITY CHECK PASSED!



In [3]:
import time

print("Clearing Data Memory...")
for i in range(20):
    data_mem.write(i*4, 0x0)

print("-" * 30)
print("FINAL VALIDATION TEST")

# HALT THE CORE (Assert Reset)
print("Asserting Reset (Core Halted)...")
reset_gpio.channel1.write(0x1, 0x0) 

# DIRTY THE MEMORY
# We can safely write now because Port B is disabled
print("Initializing Data Memory to 0xFFFFFFFF...")
data_mem.write(0x0, 0xFFFFFFFF)

# Verify write
pre_val = data_mem.read(0x0)
print(f"Pre-check Addr 0: 0x{pre_val:08x}")

# RUN PROCESSOR
# Toggle Reset 0 -> 1 to RESTART from PC=0
print("Releasing Reset (Processor Restarting)...")
reset_gpio.channel1.write(0x1, 0x1) 

# Wait for execution
time.sleep(1)

# CHECK RESULT
val = data_mem.read(0x0)
print(f"Post-check Addr 0: {val} (Hex: 0x{val:08x})")

if val == 20:
    print("\nThe System is Perfect!")
    print("(RISC-V Core executed SW instruction).")
    print("(d_we signal correctly passed to bram).")
    print("(BRAM successfully stored the data).")
elif val == 0xFFFFFFFF:
    print("\nFAIL: Processor did not run.")
else:
    print(f"\nGot 0x{val:08x} - likely problem in instruction execution.")

Clearing Data Memory...
------------------------------
FINAL VALIDATION TEST
Asserting Reset (Core Halted)...
Initializing Data Memory to 0xFFFFFFFF...
Pre-check Addr 0: 0xffffffff
Releasing Reset (Processor Restarting)...
Post-check Addr 0: 4294967295 (Hex: 0xffffffff)

FAIL: Processor did not run.


In [12]:
# --- HELPER FUNCTION ---
def load_hex_program(filename, target_memory):
    with open(filename, 'r') as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        line = line.strip()
        if not line: continue
        target_memory.write(i * 4, int(line, 16))
    print(f"Loaded {len(lines)} instructions.")

# --- EXECUTION SEQUENCE ---
print("-" * 30)
print("STARTING ULTIMATE TEST")

# 1. HALT CORE (Assert Reset)
reset_gpio.channel1.write(0x1, 0x0)
print("1. Core Halted.")

# 2. LOAD HEX (LUI-Free Version)
load_hex_program("final_test.hex", instr_mem)

# 3. CLEAR DATA MEMORY
print("2. Clearing Data Memory...")
for i in range(20):
    data_mem.write(i*4, 0x0)

# 4. RUN CORE (Release Reset)
print("3. Releasing Reset...")
reset_gpio.channel1.write(0x1, 0x1)
time.sleep(1) 

# --- VERIFICATION ---
print("-" * 30)
val_0 = data_mem.read(0x0)
val_16 = data_mem.read(0x10)

print(f"Addr 0x00: {val_0} (Expected 20)")
print(f"Addr 0x10: 0x{val_16:08x} (Expected 0xdeadb000)")

if val_16 == 0xdeadb000:
    print("\n✅ SUCCESS: Magic Value Found! The Beast is Alive.")
else:
    print("\n❌ FAIL: Still failing. Hardware is cursed.")

------------------------------
STARTING ULTIMATE TEST
1. Core Halted.
Loaded 4 instructions.
2. Clearing Data Memory...
3. Releasing Reset...
------------------------------
Addr 0x00: 0 (Expected 20)
Addr 0x10: 0x00000000 (Expected 0xdeadb000)

❌ FAIL: Still failing. Hardware is cursed.


In [14]:
print("Clearing Data Memory...")
for i in range(20):
    data_mem.write(i*4, 0x0)


Clearing Data Memory...


In [15]:
# Force 'Run' assuming ACTIVE LOW RUN (Polarity flipped)
print("Testing INVERTED polarity (GPIO 0 = Run)...")
reset_gpio.channel1.write(0, 0x1) # Force GPIO to 0
time.sleep(0.5)

val = data_mem.read(0x04)
print(f"Value at 4: {hex(val)}")

Testing INVERTED polarity (GPIO 0 = Run)...
Value at 4: 0x0


In [ ]:
from pynq import Overlay
import time

# ==========================================
# 1. SETUP
# ==========================================
overlay = Overlay("riscv_TEST_V3.bit") # Ensure this matches your file
print("Overlay Loaded.")

# Map IPs (Update names if they differ in your block design)
reset_gpio = overlay.axi_gpio_reset  
instr_mem  = overlay.bram_controller_irom
data_mem   = overlay.bram_controller_dram

# Helper Functions (Fixed for single argument read)
def write_mem(target_mem, offset, value):
    target_mem.write(offset, value)

def read_mem(target_mem, offset):
    return target_mem.read(offset)

# ==========================================
# 2. THE "SKIP-PROOF" PROGRAM
# ==========================================
# We want to catch the CPU even if the slicing is wrong.
# We will fill the memory with instructions that ALL write to different addresses.

# 0x00700093 -> addi x1, x0, 7   (x1 = 7)
# 0x00102223 -> sw   x1, 4(x0)   (Store 7 to Addr 4)
# 0x00102423 -> sw   x1, 8(x0)   (Store 7 to Addr 8)
# 0x00102623 -> sw   x1, 12(x0)  (Store 7 to Addr 12)

# If Slicing is CORRECT (Down to 2):
# Execs: ADDI, SW(4), SW(8), SW(12). Result: 7 at 4, 8, 12.

# If Slicing is WRONG (Down to 0):
# Execs: Index 0 (ADDI), Index 4 (Empty?), Index 8 (Empty?).
# Result: Memory stays 0.

print("\n--- INJECTING DIAGNOSTIC CODE ---")

# 1. Clear Data Memory (First 64 bytes)
for i in range(0, 64, 4):
    write_mem(data_mem, i, 0x00000000)

# 2. Write the Program (Instruction Memory)
# Instr 0: ADDI x1, x0, 0xDEAD (57005) -> 0xdead0093
write_mem(instr_mem, 0, 0xdead0093) 

# Instr 1: SW x1, 4(x0) -> 0x00102223
write_mem(instr_mem, 4, 0x00102223)

# Instr 4 (Offset 16): SW x1, 8(x0) -> 0x00102423
# We place a store here just in case the CPU is skipping!
write_mem(instr_mem, 16, 0x00102423)

print("Code Injected.")

# ==========================================
# 3. RUN TEST (Dual Polarity)
# ==========================================

def scan_memory():
    found_data = False
    print("  Scanning memory for writes...")
    for i in range(0, 32, 4):
        val = read_mem(data_mem, i)
        if val != 0:
            print(f"    [!] FOUND DATA at Addr {i} (Offset {hex(i)}): {hex(val)}")
            found_data = True
    if not found_data:
        print("    No data written.")

# TEST 1: Standard Polarity (GPIO 1 = Run)
print("\n--- ATTEMPT 1: GPIO 1 = RUN (Standard) ---")
reset_gpio.channel1.write(0, 0x1) # Reset
time.sleep(0.1)
reset_gpio.channel1.write(1, 0x1) # Run
time.sleep(0.5)
scan_memory()

# TEST 2: Inverted Polarity (GPIO 0 = Run)
# Only if Test 1 failed
print("\n--- ATTEMPT 2: GPIO 0 = RUN (Inverted) ---")
reset_gpio.channel1.write(1, 0x1) # Reset (Active High logic?)
time.sleep(0.1)
reset_gpio.channel1.write(0, 0x1) # Run (Active Low logic?)
time.sleep(0.5)
scan_memory()

Overlay Loaded.

--- INJECTING DIAGNOSTIC CODE ---
Code Injected.

--- ATTEMPT 1: GPIO 1 = RUN (Standard) ---
  Scanning memory for writes...
    No data written.

--- ATTEMPT 2: GPIO 0 = RUN (Inverted) ---
  Scanning memory for writes...
    No data written.
